# cap10_release_oos_poor_window_diagnostic_v1

Notebook-first diagnostic after `sparse_release_cap10_rebound_balance_v1` failed. The prior screen showed that sparse release-cap variants did not pass both held-out folds under `+30pp` relative return and `<=15%` MaxDD. This notebook does not tune another threshold. It rebuilds the same fold-local BC actors, exports daily paths, runs poor-window diagnostics with target-aware scoring, and aggregates factor differences to decide what mechanism should come next.

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_release_cap10_rebound_balance_v1`.
- **Role**: `diagnose`.
- **Changed layer**: reusable `oos_diagnostics` scoring plus diagnostic notebook only.
- **Target windows**: `wf_2022` held out in 2022; `wf_2023_2024` held out in 2023-2024.
- **Acceptance criteria**: identify dominant failure type, top poor-window clusters, top feature-delta hypotheses, and a no-promotion next mechanism.
- **Failure mode**: if daily-path diagnostics cannot distinguish drawdown control from trend/participation failure, do not launch PPO or a new reward sweep.

External basis retained for this design:

- Qlib experiment/replay framework: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- Backtest overfitting / CSCV discipline: https://doi.org/10.21314/JCF.2016.322
- Deflated Sharpe / multiple-testing caution: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551
- Purged and embargoed cross-validation framing: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678


In [1]:

from __future__ import annotations

import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.dashboard import write_review_daily_csv
from src.soxl_quant.app.oos_diagnostics import (
    aggregate_feature_differences,
    aggregate_poor_windows,
    attribute_gate_gaps,
    normalise_gate_metrics_from_selection,
    run_oos_diagnostic,
    summarize_gate_gap_attribution,
)
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'cap10_release_oos_poor_window_diagnostic_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
DIRECT_CARD = ROOT / 'runtime/model_selection/card/sparse_release_cap10_rebound_balance_v1'
SOURCE_CARD = ROOT / 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1'
TARGETS_PATH = SOURCE_CARD / 'sparse_release_revised_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
RISK_ANCHOR_CAP = 0.10
RELEASE_CAP_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
WINDOW_SIZE = 63
TOP_WINDOWS = 8
TOP_FEATURES = 24
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / 'daily_paths'
DIAG_DIR = CARD / 'diagnostics'
AGG_DIR = CARD / 'aggregate'
for path in [DAILY_DIR, DIAG_DIR, AGG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'source_target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'source_target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2021-12-31'),
        'valid': ('2022-01-01', '2022-12-31'),
        'test': ('2023-01-01', '2024-12-31'),
    },
}

EXTERNAL_REFERENCES = [
    {
        'label': 'Qlib GitHub',
        'url': 'https://github.com/microsoft/qlib',
        'note': 'keep the replay/backtest workflow inside the qlib harness',
    },
    {
        'label': 'Qlib arXiv',
        'url': 'https://arxiv.org/abs/2009.11189',
        'note': 'AI quant platform scope for data, modeling, backtest, and deployment',
    },
    {
        'label': 'Probability of Backtest Overfitting',
        'url': 'https://doi.org/10.21314/JCF.2016.322',
        'note': 'avoid selecting a candidate from repeated screens without OOS discipline',
    },
    {
        'label': 'Deflated Sharpe Ratio',
        'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551',
        'note': 'multiple-testing caution for model and factor screens',
    },
]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def as_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.map(lambda v: False if pd.isna(v) else str(v).strip().lower() in {'true', '1', 'yes', 'y'}).astype(bool)


def clean_target_frame(targets: pd.DataFrame) -> pd.DataFrame:
    frame = targets.copy()
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    for col in ['teacher_position', 'rebound_target_position', 'risk_target_position', 'objective_weight']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').fillna(0.0)
    for col in ['is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'release_gate_predicted']:
        if col in frame.columns:
            frame[col] = as_bool(frame[col])
    if 'release_gate_state' not in frame.columns:
        frame['release_gate_state'] = False
    return frame


def build_release_cap_targets(base_targets: pd.DataFrame, *, release_cap: float) -> pd.DataFrame:
    targets = clean_target_frame(base_targets)
    targets['source_risk_target_position'] = targets['risk_target_position']
    targets['risk_target_position_original'] = float(RISK_ANCHOR_CAP)
    targets['risk_anchor_cap'] = float(RISK_ANCHOR_CAP)
    targets['release_cap'] = float(release_cap)
    targets['target_variant'] = f'cap10_release_{release_cap:.2f}'
    downtrend = targets['is_downtrend_state'].astype(bool)
    release = targets['release_gate_state'].astype(bool) & downtrend
    targets.loc[downtrend, 'risk_target_position'] = float(RISK_ANCHOR_CAP)
    release_position = np.maximum(
        float(RISK_ANCHOR_CAP),
        np.minimum(targets['rebound_target_position'].to_numpy(dtype=float), float(release_cap)),
    )
    targets.loc[release, 'risk_target_position'] = release_position[release.to_numpy()]
    targets['release_cap_effective_lift'] = targets['risk_target_position'] - targets['risk_target_position_original']
    targets['release_cap_active'] = (release & (targets['release_cap_effective_lift'] > 1e-9)).astype(bool)
    return targets.reset_index(drop=True)


def evaluate_actor_metrics(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> tuple[dict[str, float], dict[str, object]]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    return out, bt


def fold_label(release_cap: float, fold_name: str) -> str:
    return f'cap10_release_{release_cap:.2f}__{fold_name}'


In [3]:

profile = load_profile(PROFILE_NAME)
direct_selection = json.loads((DIRECT_CARD / 'selection.json').read_text(encoding='utf-8'))
direct_summary = json.loads((DIRECT_CARD / 'summary.json').read_text(encoding='utf-8'))
base_targets = clean_target_frame(pd.read_csv(TARGETS_PATH))
frame = pd.read_pickle(FRAME_PATH)
prior_fold_results = pd.read_csv(DIRECT_CARD / 'target_variant_fold_results.csv')
prior_summary = pd.read_csv(DIRECT_CARD / 'target_variant_summary.csv')

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    fold_targets = base_targets[base_targets['fold'].astype(str).eq(spec['source_target_fold'])]
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    eval_overlap = fold_targets[(fold_targets['datetime'] >= test_start) & (fold_targets['datetime'] <= test_end)]
    coverage_rows.append({
        'fold': fold_name,
        'source_target_fold': spec['source_target_fold'],
        'target_rows': int(len(fold_targets)),
        'target_start': str(fold_targets['datetime'].min().date()) if len(fold_targets) else '',
        'target_end': str(fold_targets['datetime'].max().date()) if len(fold_targets) else '',
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'eval_overlap_rows': int(len(eval_overlap)),
        'fold_local_gate': bool(len(fold_targets) > 0 and len(eval_overlap) == 0),
    })
coverage = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'direct_comparison_experiment': 'sparse_release_cap10_rebound_balance_v1',
    'direct_comparison_decision': direct_selection.get('decision'),
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'risk_anchor_cap': RISK_ANCHOR_CAP,
    'release_cap_grid': RELEASE_CAP_GRID,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'window_size': WINDOW_SIZE,
    'windows': coverage_rows,
}
pd.DataFrame(coverage_rows).to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', coverage)
display(pd.DataFrame(coverage_rows))
display(prior_summary)


,fold,source_target_fold,target_rows,target_start,target_end,test_start,test_end,eval_overlap_rows,fold_local_gate
0,wf_2022,wf_2022_train,505,2020-01-02,2021-12-31,2022-01-01,2022-12-31,0,True
1,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,2023-01-01,2024-12-31,0,True


,target_variant,release_cap,folds_evaluated,fold_gate_pass_count,all_fold_gate,slice_gate_pass_count,all_slice_gate,participation_gate,mean_target_score,worst_target_score,mean_relative_return,worst_max_dd,mean_avg_pos,min_release_accuracy,min_downtrend_accuracy,strict_gate
0,cap10_release_0.10,0.10,2,1,False,2,True,True,24.692751,49.385503,26.575972,12.291345,0.150793,1.000,0.968553,False
1,cap10_release_0.15,0.15,2,1,False,2,True,True,25.201626,50.403253,25.766610,13.470361,0.142767,0.975,0.952991,False
2,cap10_release_0.20,0.20,2,1,False,1,False,True,27.489779,54.979558,24.065299,11.401851,0.143395,0.025,0.872117,False
3,cap10_release_0.35,0.35,2,1,False,1,False,False,27.835320,54.670640,24.937429,13.631561,0.104374,0.000,0.901468,False
4,cap10_release_0.30,0.30,2,1,False,1,False,False,28.251784,55.848306,25.794454,12.291345,0.112959,0.000,0.884615,False
5,cap10_release_0.25,0.25,2,0,False,1,False,True,50.756001,54.287107,17.968874,26.806224,0.145686,0.000,0.893162,False


In [4]:

init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=220, lr=0.001, batch_size=128, seed=SEED)
weight_config = StateObjectiveBCSampleWeightConfig(
    downtrend_multiplier=2.0,
    rebound_multiplier=2.0,
    ordinary_multiplier=1.0,
    objective_weight_power=0.5,
    class_balanced_beta=0.99,
    weight_cap=20.0,
)

fold_datasets = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    fold_datasets[fold_name] = {
        'dataset': dataset,
        'test_df': segment_frame(dataset, 'test'),
        'test_price': price_frame.loc[spec['test'][0]:spec['test'][1]],
    }

metric_rows = []
train_rows = []
daily_rows = []
windows_by_label = {}
features_by_label = {}

for variant_idx, release_cap in enumerate(RELEASE_CAP_GRID):
    variant_name = f'cap10_release_{release_cap:.2f}'
    variant_targets = build_release_cap_targets(base_targets, release_cap=release_cap)
    for fold_idx, (fold_name, spec) in enumerate(WALK_FORWARD_WINDOWS.items()):
        label = fold_label(release_cap, fold_name)
        fold_targets = variant_targets[variant_targets['fold'].astype(str).eq(spec['source_target_fold'])].copy()
        seed = SEED + variant_idx * 100 + fold_idx
        set_seed(seed)
        bc_result = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=StateObjectiveBCTrainingConfig(
                steps=train_config.steps,
                lr=train_config.lr,
                batch_size=train_config.batch_size,
                seed=seed,
            ),
            sample_weight_config=weight_config,
            sample_size=0,
            rng_seed=seed,
        )
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=seed,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            bc_result.actor,
            bc_result.obs,
            bc_result.actions,
            bc_dataset.metadata,
            slice_cols=('release_gate_state', 'is_downtrend_state', 'is_rebound_cluster'),
        )
        actor = bc_result.actor
        actor.eval()
        data = fold_datasets[fold_name]
        metrics, bt = evaluate_actor_metrics(actor, data['test_df'], data['test_price'], profile, spec['test'])
        daily_path = DAILY_DIR / f'{label}.csv'
        write_review_daily_csv(
            daily_path,
            bt['portfolio'],
            data['test_price'],
            bt.get('position_history', []),
            bt.get('trade_signals', []),
        )
        diag_out = DIAG_DIR / label
        manifest = run_oos_diagnostic(
            daily_csv=daily_path,
            profile=label,
            output_dir=diag_out,
            oos_start=spec['test'][0],
            oos_end=spec['test'][1],
            window_size=WINDOW_SIZE,
            top_windows=TOP_WINDOWS,
            top_features=TOP_FEATURES,
            target_excess=TARGET_RELATIVE_RETURN,
            target_max_dd=TARGET_MAX_DD,
            train_end=spec['train'][1],
            external_references=EXTERNAL_REFERENCES,
        )
        poor_windows = pd.read_csv(diag_out / 'poor_windows.csv')
        feature_diffs = pd.read_csv(diag_out / 'feature_differences.csv')
        windows_by_label[label] = poor_windows
        features_by_label[label] = feature_diffs
        daily_rows.append({
            'label': label,
            'target_variant': variant_name,
            'release_cap': float(release_cap),
            'fold': fold_name,
            'daily_csv': str(daily_path.relative_to(ROOT)),
            'diagnostic_dir': str(diag_out.relative_to(ROOT)),
            'poor_windows': int(manifest['windows']),
            'feature_differences': int(manifest['features']),
        })
        metric_rows.append({
            'label': label,
            'target_variant': variant_name,
            'release_cap': float(release_cap),
            'fold': fold_name,
            'return': float(metrics.get('return', np.nan)),
            'buy_and_hold_return': float(metrics.get('buy_and_hold_return', np.nan)),
            'relative_return': float(metrics.get('relative_return', np.nan)),
            'sharpe': float(metrics.get('sharpe', np.nan)),
            'max_dd': float(metrics.get('max_dd', np.nan)),
            'avg_pos': float(metrics.get('avg_pos', np.nan)),
            'gate_pass': bool(
                float(metrics.get('relative_return', -1e9)) >= TARGET_RELATIVE_RETURN
                and float(metrics.get('max_dd', 1e9)) <= TARGET_MAX_DD
            ),
        })
        train_rows.append({
            'label': label,
            'target_variant': variant_name,
            'release_cap': float(release_cap),
            'fold': fold_name,
            'target_rows': int(len(fold_targets)),
            'release_gate_rows': int(fold_targets['release_gate_state'].sum()),
            'active_release_lift_rows': int(fold_targets['release_cap_active'].sum()),
            'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
            'bc_after_loss': float(bc_result.metrics['after_loss']),
            'release_slice_accuracy': float(slice_metrics.set_index('slice').loc['release_gate_state', 'accuracy']) if 'release_gate_state' in set(slice_metrics['slice']) else np.nan,
            'downtrend_slice_accuracy': float(slice_metrics.set_index('slice').loc['is_downtrend_state', 'accuracy']) if 'is_downtrend_state' in set(slice_metrics['slice']) else np.nan,
        })

path_metrics = pd.DataFrame(metric_rows)
train_metrics = pd.DataFrame(train_rows)
daily_manifest = pd.DataFrame(daily_rows)
path_metrics.to_csv(CARD / 'daily_path_metrics.csv', index=False)
train_metrics.to_csv(CARD / 'bc_train_metrics.csv', index=False)
daily_manifest.to_csv(CARD / 'daily_path_manifest.csv', index=False)
display(path_metrics)
display(daily_manifest)


[28965:MainThread](2026-05-26 05:46:50,453) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[28965:MainThread](2026-05-26 05:46:51,167) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[28965:MainThread](2026-05-26 05:46:51,176) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[28965:MainThread](2026-05-26 05:46:53,342) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[28965:MainThread](2026-05-26 05:46:53,343) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[28965:MainThread](2026-05-26 05:49:38,462) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[28965:MainThread](2026-05-26 05:49:38,464) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[28965:MainThread](2026-05-26 05:49:38,465) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


,label,target_variant,release_cap,fold,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,gate_pass
0,cap10_release_0.10__wf_2022,cap10_release_0.10,0.10,wf_2022,-10.137677,-76.614257,72.537447,-0.767384,11.809271,0.171053,True
1,cap10_release_0.10__wf_2023_2024,cap10_release_0.10,0.10,wf_2023_2024,20.436335,55.654474,-19.385503,0.889839,10.276470,0.130533,False
2,cap10_release_0.15__wf_2022,cap10_release_0.15,0.15,wf_2022,-1.203459,-76.614257,71.918219,-0.012868,10.014948,0.156447,True
3,cap10_release_0.15__wf_2023_2024,cap10_release_0.15,0.15,wf_2023_2024,19.497958,55.654474,-20.384999,0.865815,10.990701,0.129087,False
4,cap10_release_0.20__wf_2022,cap10_release_0.20,0.20,wf_2022,-1.027383,-76.614257,68.074895,0.015813,12.051659,0.158553,True
5,cap10_release_0.20__wf_2023_2024,cap10_release_0.20,0.20,wf_2023_2024,28.989484,55.654474,-19.944298,1.261319,8.851442,0.128237,False
6,cap10_release_0.25__wf_2022,cap10_release_0.25,0.25,wf_2022,-12.260703,-76.614257,55.207557,-0.584832,19.092674,0.162237,False
7,cap10_release_0.25__wf_2023_2024,cap10_release_0.25,0.25,wf_2023_2024,28.070650,55.654474,-19.269810,1.230196,8.851442,0.129135,False
8,cap10_release_0.30__wf_2022,cap10_release_0.30,0.30,wf_2022,-10.649173,-76.614257,72.410846,-0.904598,12.251923,0.097237,True
9,cap10_release_0.30__wf_2023_2024,cap10_release_0.30,0.30,wf_2023_2024,20.436335,55.654474,-20.821938,0.889839,10.276470,0.128682,False


,label,target_variant,release_cap,fold,daily_csv,diagnostic_dir,poor_windows,feature_differences
0,cap10_release_0.10__wf_2022,cap10_release_0.10,0.10,wf_2022,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
1,cap10_release_0.10__wf_2023_2024,cap10_release_0.10,0.10,wf_2023_2024,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
2,cap10_release_0.15__wf_2022,cap10_release_0.15,0.15,wf_2022,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
3,cap10_release_0.15__wf_2023_2024,cap10_release_0.15,0.15,wf_2023_2024,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
4,cap10_release_0.20__wf_2022,cap10_release_0.20,0.20,wf_2022,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
5,cap10_release_0.20__wf_2023_2024,cap10_release_0.20,0.20,wf_2023_2024,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
6,cap10_release_0.25__wf_2022,cap10_release_0.25,0.25,wf_2022,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
7,cap10_release_0.25__wf_2023_2024,cap10_release_0.25,0.25,wf_2023_2024,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
8,cap10_release_0.30__wf_2022,cap10_release_0.30,0.30,wf_2022,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24
9,cap10_release_0.30__wf_2023_2024,cap10_release_0.30,0.30,wf_2023_2024,runtime/model_selection/card/cap10_release_oos...,runtime/model_selection/card/cap10_release_oos...,8,24


In [5]:

poor_window_clusters = aggregate_poor_windows(
    windows_by_label,
    target_excess=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
    overlap_gap_days=5,
)
feature_delta_clusters = aggregate_feature_differences(features_by_label, top_n=40)
metrics_by_label = {
    row['label']: {'heldout': row.drop(labels=['label', 'target_variant', 'release_cap', 'fold', 'gate_pass']).to_dict()}
    for _, row in path_metrics.iterrows()
}
gate_gaps = attribute_gate_gaps(
    metrics_by_label,
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
gate_summary = summarize_gate_gap_attribution(gate_gaps)

poor_window_clusters.to_csv(AGG_DIR / 'poor_window_clusters.csv', index=False)
feature_delta_clusters.to_csv(AGG_DIR / 'feature_delta_clusters.csv', index=False)
gate_gaps.to_csv(AGG_DIR / 'gate_gap_attribution.csv', index=False)
gate_summary.to_csv(AGG_DIR / 'gate_gap_profile_summary.csv', index=False)

# Notebook-specific, non-shared grouping to make feature hypotheses readable.
def feature_group(name: str) -> str:
    upper = name.upper()
    if 'SOXX' in upper or 'RELATIVE' in upper or 'TRACKING' in upper or 'TE_' in upper:
        return 'soxx_relative_or_tracking_error'
    if 'VIX' in upper or 'VOL' in upper or 'KURT' in upper or 'DOWNSIDE' in upper:
        return 'volatility_or_downside'
    if 'NEW_LOW' in upper or 'REGIME' in upper or 'RISK' in upper or 'DRAWDOWN' in upper or 'DD' in upper:
        return 'regime_or_drawdown'
    if (
        'TREND' in upper
        or 'MA' in upper
        or 'MOM' in upper
        or 'RETURN' in upper
        or 'ANCHOR' in upper
        or 'EXPECTANCY' in upper
        or 'SHARPE' in upper
    ):
        return 'trend_or_return'
    if 'VOLUME' in upper or 'CICC' in upper:
        return 'volume_or_flow'
    return 'other'

feature_group_consensus = feature_delta_clusters.copy()
if not feature_group_consensus.empty:
    feature_group_consensus['feature_group'] = feature_group_consensus['feature'].map(feature_group)
    feature_group_consensus = (
        feature_group_consensus.groupby('feature_group', as_index=False)
        .agg(
            feature_count=('feature', 'count'),
            mean_abs_delta_z=('mean_abs_delta_z', 'mean'),
            max_stability_score=('stability_score', 'max'),
            mean_spearman_to_excess=('mean_spearman_to_excess', 'mean'),
            top_features=('feature', lambda values: ','.join(list(values)[:6])),
        )
        .sort_values(['max_stability_score', 'mean_abs_delta_z'], ascending=[False, False])
        .reset_index(drop=True)
    )
feature_group_consensus.to_csv(AGG_DIR / 'feature_group_consensus.csv', index=False)

display(poor_window_clusters.head(12))
display(gate_summary.head(12))
display(feature_group_consensus.head(12))


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C003,2023-10-23,2024-03-07,6,48,"cap10_release_0.10__wf_2023_2024,cap10_release...",-112.967255,-130.537576,5.151819,5.151819,0.106834,142.967255,160.537576,underparticipation,reentry_participation_or_behavior_clone
1,C002,2022-09-02,2022-12-29,6,33,"cap10_release_0.10__wf_2022,cap10_release_0.15...",-8.070640,-17.893409,6.808061,9.060112,0.018385,38.070640,47.893409,underparticipation,reentry_participation_or_behavior_clone
2,C001,2022-05-09,2022-08-18,5,15,"cap10_release_0.10__wf_2022,cap10_release_0.15...",-4.818807,-10.913696,8.937712,14.739362,0.080653,34.818807,40.913696,underparticipation,reentry_participation_or_behavior_clone


,profile,window_count,gate_pass_count,failure_count,dominant_failure_type,worst_window,max_severity_score,recommended_next_mechanism
0,cap10_release_0.30__wf_2023_2024,1,0,1,underparticipation,heldout,50.821938,participation_or_reentry_objective
1,cap10_release_0.15__wf_2023_2024,1,0,1,underparticipation,heldout,50.384999,participation_or_reentry_objective
2,cap10_release_0.20__wf_2023_2024,1,0,1,underparticipation,heldout,49.944298,participation_or_reentry_objective
3,cap10_release_0.35__wf_2023_2024,1,0,1,underparticipation,heldout,49.645584,participation_or_reentry_objective
4,cap10_release_0.10__wf_2023_2024,1,0,1,underparticipation,heldout,49.385503,participation_or_reentry_objective
5,cap10_release_0.25__wf_2023_2024,1,0,1,underparticipation,heldout,49.269810,participation_or_reentry_objective
6,cap10_release_0.25__wf_2022,1,0,1,drawdown_control,heldout,40.926744,drawdown_controller_or_sequence_risk_feature
7,cap10_release_0.10__wf_2022,1,1,0,pass,heldout,0.000000,none
8,cap10_release_0.15__wf_2022,1,1,0,pass,heldout,0.000000,none
9,cap10_release_0.20__wf_2022,1,1,0,pass,heldout,0.000000,none


,feature_group,feature_count,mean_abs_delta_z,max_stability_score,mean_spearman_to_excess,top_features
0,regime_or_drawdown,6,0.947317,4.694082,-0.017558,"rolling_new_low_ratio_63,regime_risk,drawdown_..."
1,volatility_or_downside,12,0.870869,4.478630,0.079657,"realized_kurtosis_21,vol_coef_21,regime_risk_n..."
2,trend_or_return,12,0.851141,3.490700,-0.057655,"anchor_slope_180,below_anchor_ratio_63,anchor_..."
3,soxx_relative_or_tracking_error,3,0.802526,2.467018,-0.008355,"tracking_error_vol_21,relative_strength_21,tra..."
4,other,7,0.695407,1.608011,-0.107401,"cum_drop_30,rsi_14,cum_drop_20,days_since_high..."


In [6]:

if gate_summary.empty:
    dominant_gate_failure = 'none'
    worst_label = ''
    recommended_next = 'collect_daily_path_diagnostics'
else:
    severity_by_type = gate_gaps[~gate_gaps['gate_pass'].astype(bool)].groupby('failure_type')['severity_score'].sum().sort_values(ascending=False)
    dominant_gate_failure = str(severity_by_type.index[0]) if not severity_by_type.empty else 'pass'
    worst_label = str(gate_gaps.sort_values('severity_score', ascending=False).iloc[0]['profile'])
    recommended_next = str(gate_summary.sort_values('max_severity_score', ascending=False).iloc[0]['recommended_next_mechanism'])

if poor_window_clusters.empty:
    dominant_cluster_failure = 'none'
else:
    cluster_severity = poor_window_clusters.groupby('dominant_failure_type')['max_bad_score'].sum().sort_values(ascending=False)
    dominant_cluster_failure = str(cluster_severity.index[0])

if dominant_gate_failure in {'underparticipation', 'trend_capture'} or dominant_cluster_failure == 'underparticipation':
    decision = 'diagnose_2023_2024_trend_participation_before_training'
    next_mechanism = 'state_conditioned_trend_participation_with_drawdown_controller_screen'
elif dominant_gate_failure in {'drawdown_control', 'joint_tradeoff'} or dominant_cluster_failure == 'drawdown_control':
    decision = 'diagnose_joint_drawdown_participation_tradeoff_before_training'
    next_mechanism = 'walkforward_weighted_multi_objective_training_target_screen'
else:
    decision = 'diagnostic_inconclusive_no_training'
    next_mechanism = 'collect_additional_daily_paths_or_factor_ic_before_training'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'sparse_release_cap10_rebound_balance_v1',
    'role': 'diagnose',
    'changed_layer': ['diagnostic', 'model_selection'],
    'promotion_allowed': False,
    'promotion_blocker': 'Daily-path poor-window diagnostics are explanatory only; no actor or threshold is promoted.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'window_size': WINDOW_SIZE,
    'release_cap_grid': RELEASE_CAP_GRID,
    'dominant_gate_failure': dominant_gate_failure,
    'dominant_cluster_failure': dominant_cluster_failure,
    'worst_label': worst_label,
    'daily_path_count': int(len(path_metrics)),
    'poor_window_cluster_count': int(len(poor_window_clusters)),
    'feature_delta_cluster_count': int(len(feature_delta_clusters)),
    'top_feature_groups': feature_group_consensus.head(5).to_dict(orient='records') if not feature_group_consensus.empty else [],
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'daily_path_manifest.csv',
        'daily_path_metrics.csv',
        'bc_train_metrics.csv',
        'aggregate/poor_window_clusters.csv',
        'aggregate/feature_delta_clusters.csv',
        'aggregate/feature_group_consensus.csv',
        'aggregate/gate_gap_attribution.csv',
        'aggregate/gate_gap_profile_summary.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [ref['url'] for ref in EXTERNAL_REFERENCES],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'coverage': coverage,
    'direct_comparison': {
        'experiment': 'sparse_release_cap10_rebound_balance_v1',
        'decision': direct_selection.get('decision'),
        'conclusion': direct_summary.get('conclusion', []),
    },
    'gate_summary': gate_summary.to_dict(orient='records'),
    'poor_window_clusters': poor_window_clusters.head(20).to_dict(orient='records'),
    'feature_group_consensus': feature_group_consensus.to_dict(orient='records') if not feature_group_consensus.empty else [],
    'conclusion': [],
}
summary['conclusion'].append(
    f"Dominant gate failure is {dominant_gate_failure}; dominant rolling poor-window cluster failure is {dominant_cluster_failure}."
)
if not poor_window_clusters.empty:
    top = poor_window_clusters.iloc[0]
    summary['conclusion'].append(
        f"Top cluster {top['cluster_id']} spans {top['cluster_start']}..{top['cluster_end']} across {top['profile_count']} profile labels; "
        f"dominant={top['dominant_failure_type']}, mean_avg_pos={float(top['mean_avg_pos']):.3f}, worst_excess={float(top['worst_excess_return']):.2f}pp."
    )
if not feature_group_consensus.empty:
    top_group = feature_group_consensus.iloc[0]
    summary['conclusion'].append(
        f"Top feature-delta group is {top_group['feature_group']} with features {top_group['top_features']}; treat as hypothesis source only, not a promoted factor."
    )
summary['conclusion'].append(
    f"Decision: {decision}. Next mechanism: {next_mechanism}. No promotion."
)

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
summary


{'experiment': 'cap10_release_oos_poor_window_diagnostic_v1',
 'decision': 'diagnose_2023_2024_trend_participation_before_training',
 'selection': {'experiment': 'cap10_release_oos_poor_window_diagnostic_v1',
  'created_at': '2026-05-25T21:49:38.918815+00:00',
  'decision': 'diagnose_2023_2024_trend_participation_before_training',
  'next_mechanism': 'state_conditioned_trend_participation_with_drawdown_controller_screen',
  'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
  'direct_comparison_baseline': 'sparse_release_cap10_rebound_balance_v1',
  'role': 'diagnose',
  'changed_layer': ['diagnostic', 'model_selection'],
  'promotion_allowed': False,
  'promotion_blocker': 'Daily-path poor-window diagnostics are explanatory only; no actor or threshold is promoted.',
  'target_relative_return_pp': 30.0,
  'target_max_dd_pct': 15.0,
  'window_size': 63,
  'release_cap_grid': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35],
  'dominant_gate_failure': 'underparticipation',
  'dominant_clust

In [7]:

lines = [
    f"## Runtime Conclusion: {EXPERIMENT}",
    "",
    f"- **Decision**: `{selection['decision']}`.",
    f"- **Promotion**: blocked. {selection['promotion_blocker']}",
    f"- **Dominant gate failure**: `{selection['dominant_gate_failure']}`.",
    f"- **Dominant rolling-window failure**: `{selection['dominant_cluster_failure']}`.",
    f"- **Daily paths diagnosed**: {selection['daily_path_count']}.",
    f"- **Poor-window clusters**: {selection['poor_window_cluster_count']}.",
    "",
    "### Readout",
]
for item in summary['conclusion']:
    lines.append(f"- {item}")
lines += ["", "### Top Gate Gaps"]
for row in gate_gaps.head(8).to_dict(orient='records'):
    lines.append(
        f"- `{row['profile']}`: type={row['failure_type']}, rel={row['relative_return']:.2f}pp, "
        f"MaxDD={row['max_dd']:.2f}%, avg_pos={row['avg_pos']:.3f}, path={row['primary_optimization_path']}."
    )
lines += ["", "### Top Feature Groups"]
if feature_group_consensus.empty:
    lines.append("- none")
else:
    for row in feature_group_consensus.head(5).to_dict(orient='records'):
        lines.append(
            f"- `{row['feature_group']}`: features={row['top_features']}, "
            f"score={row['max_stability_score']:.3f}, spearman={row['mean_spearman_to_excess']:.3f}."
        )
lines += ["", "### Artifacts"]
for sidecar in selection['sidecars']:
    lines.append(f"- `{sidecar}`")
lines += ["", "### Next Action", f"- `{selection['next_mechanism']}`."]
display(Markdown("\n".join(lines)))


## Runtime Conclusion: cap10_release_oos_poor_window_diagnostic_v1

- **Decision**: `diagnose_2023_2024_trend_participation_before_training`.
- **Promotion**: blocked. Daily-path poor-window diagnostics are explanatory only; no actor or threshold is promoted.
- **Dominant gate failure**: `underparticipation`.
- **Dominant rolling-window failure**: `underparticipation`.
- **Daily paths diagnosed**: 12.
- **Poor-window clusters**: 3.

### Readout
- Dominant gate failure is underparticipation; dominant rolling poor-window cluster failure is underparticipation.
- Top cluster C003 spans 2023-10-23..2024-03-07 across 6 profile labels; dominant=underparticipation, mean_avg_pos=0.107, worst_excess=-130.54pp.
- Top feature-delta group is regime_or_drawdown with features rolling_new_low_ratio_63,regime_risk,drawdown_63,drawdown_21,new_low_ratio_21,regime_risk_velocity_5; treat as hypothesis source only, not a promoted factor.
- Decision: diagnose_2023_2024_trend_participation_before_training. Next mechanism: state_conditioned_trend_participation_with_drawdown_controller_screen. No promotion.

### Top Gate Gaps
- `cap10_release_0.30__wf_2023_2024`: type=underparticipation, rel=-20.82pp, MaxDD=10.28%, avg_pos=0.129, path=participation_or_reentry_objective.
- `cap10_release_0.15__wf_2023_2024`: type=underparticipation, rel=-20.38pp, MaxDD=10.99%, avg_pos=0.129, path=participation_or_reentry_objective.
- `cap10_release_0.20__wf_2023_2024`: type=underparticipation, rel=-19.94pp, MaxDD=8.85%, avg_pos=0.128, path=participation_or_reentry_objective.
- `cap10_release_0.35__wf_2023_2024`: type=underparticipation, rel=-19.65pp, MaxDD=10.28%, avg_pos=0.129, path=participation_or_reentry_objective.
- `cap10_release_0.10__wf_2023_2024`: type=underparticipation, rel=-19.39pp, MaxDD=10.28%, avg_pos=0.131, path=participation_or_reentry_objective.
- `cap10_release_0.25__wf_2023_2024`: type=underparticipation, rel=-19.27pp, MaxDD=8.85%, avg_pos=0.129, path=participation_or_reentry_objective.
- `cap10_release_0.25__wf_2022`: type=drawdown_control, rel=55.21pp, MaxDD=19.09%, avg_pos=0.162, path=drawdown_controller_or_sequence_risk_feature.
- `cap10_release_0.10__wf_2022`: type=pass, rel=72.54pp, MaxDD=11.81%, avg_pos=0.171, path=none.

### Top Feature Groups
- `regime_or_drawdown`: features=rolling_new_low_ratio_63,regime_risk,drawdown_63,drawdown_21,new_low_ratio_21,regime_risk_velocity_5, score=4.694, spearman=-0.018.
- `volatility_or_downside`: features=realized_kurtosis_21,vol_coef_21,regime_risk_no_vix,vix_level,vix_percentile_252,downside_semivol_21, score=4.479, spearman=0.080.
- `trend_or_return`: features=anchor_slope_180,below_anchor_ratio_63,anchor_gap_180,expectancy_proxy_21,sharpe_proxy_21,mom_21, score=3.491, spearman=-0.058.
- `soxx_relative_or_tracking_error`: features=tracking_error_vol_21,relative_strength_21,tracking_error_21, score=2.467, spearman=-0.008.
- `other`: features=cum_drop_30,rsi_14,cum_drop_20,days_since_high_63,realized_skew_21,jump_variation_5, score=1.608, spearman=-0.107.

### Artifacts
- `coverage.csv`
- `coverage.json`
- `daily_path_manifest.csv`
- `daily_path_metrics.csv`
- `bc_train_metrics.csv`
- `aggregate/poor_window_clusters.csv`
- `aggregate/feature_delta_clusters.csv`
- `aggregate/feature_group_consensus.csv`
- `aggregate/gate_gap_attribution.csv`
- `aggregate/gate_gap_profile_summary.csv`
- `selection.json`
- `summary.json`

### Next Action
- `state_conditioned_trend_participation_with_drawdown_controller_screen`.